# RQ4: Toxic Discourse

Standalone Perspective API toxicity workflow. Requires PERSPECTIVE_API_KEY in the environment or .env file.


In [ ]:
DATA_DIR = 'C:/Projects/israel_hamas_discourse_analysis/data'
SENTIMENT_OUTPUT_DIR = 'C:/Projects/israel_hamas_discourse_analysis/02_emotional_tone_analysis/outputs'
OUTPUT_DIR = 'C:/Projects/israel_hamas_discourse_analysis/05_toxicity_analysis/outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_labeled_data():
    reddit = pd.read_csv(f'{DATA_DIR}/reddit_labeled_cleaned.csv')
    youtube = pd.read_csv(f'{DATA_DIR}/youtube_labeled_cleaned.csv')

    for col in ['score', 'post_score', 'post_upvote_ratio', 'user_total_karma', 'controversiality']:
        if col in reddit.columns:
            reddit[col] = pd.to_numeric(reddit[col], errors='coerce')
    for col in ['likeCount', 'replyCount']:
        if col in youtube.columns:
            youtube[col] = pd.to_numeric(youtube[col], errors='coerce')

In [ ]:
from pathlib import Path

# ==================== EDIT THESE PATHS ====================
# Local mode:  INPUT_DIR = Path('../../02_emotional_tone_analysis/outputs')
# Kaggle mode: INPUT_DIR = Path('/kaggle/input/israel-hamas-data')
# =========================================================

INPUT_DIR = Path('../../02_emotional_tone_analysis/outputs')  # <-- EDIT: Module 02 outputs (reddit_with_sentiment.csv, etc.)
OUTPUT_DIR = Path.cwd()                                        # <-- EDIT: where to save outputs

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input directory:  {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## Kaggle Setup & Input Paths

Configure paths and load sentiment data from Module 02. (Note: Perspective API requires API key setup in Kaggle environment)



## Perspective API Toxicity Analysis


In [ ]:
"""
Phase 6: Perspective API Analysis
Critical analysis of discourse toxicity and harmful content using Google's Perspective API.
Attributes analyzed: TOXICITY, SEVERE_TOXICITY, IDENTITY_ATTACK, THREAT, INSULT, PROFANITY
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from googleapiclient import discovery
import json
import time
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create output directory

print("="*80)
print("PHASE 6: PERSPECTIVE API CRITICAL ANALYSIS")
print("="*80)


## 1. Setup API Client

In [ ]:
# ============================================================================
API_KEY = os.getenv('PERSPECTIVE_API_KEY')

if not API_KEY:
    raise ValueError("API Key not found. Please set PERSPECTIVE_API_KEY in .env file")

client = discovery.build(
    "commentanalyzer",
    "v1alpha1",
    developerKey=API_KEY,
    discoveryServiceUrl="https://commentanalyzer.googleapis.com/$discovery/rest?version=v1alpha1",
    static_discovery=False,
)

def get_perspective_scores(text):
    """
    Get scores for multiple attributes from Perspective API
    """
    if pd.isna(text) or text == '' or len(str(text)) < 2:
        return None
        
    analyze_request = {
        'comment': {'text': str(text)[:3000]}, # Limit length to avoid errors
        'requestedAttributes': {
            'TOXICITY': {},
            'SEVERE_TOXICITY': {},
            'IDENTITY_ATTACK': {},
            'INSULT': {},
            'THREAT': {},
            'PROFANITY': {}
        },
        'languages': ['en']
    }
    
    try:
        response = client.comments().analyze(body=analyze_request).execute()
        scores = {}
        for attr in response['attributeScores']:
            scores[attr] = response['attributeScores'][attr]['summaryScore']['value']
        return scores
    except Exception as e:
        # print(f"Error: {e}") # Suppress individual errors to keep output clean
        return None


## 2. Load Data

In [ ]:
# ============================================================================
print("\nLoading data...")
reddit_df, youtube_df = load_sentiment_data()

# Filter valid labels
valid_labels = ['P', 'I', 'N']
reddit_df = reddit_df[reddit_df['Label'].isin(valid_labels)]
youtube_df = youtube_df[youtube_df['Label'].isin(valid_labels)]

# Sample data to stay within API quotas (Perspective has rate limits)
# Let's take a stratified sample of 500 from each platform for demonstration/analysis
# In a full run, you might want to run this in batches over time.
SAMPLE_SIZE = 300 

print(f"Sampling {SAMPLE_SIZE} comments per platform for API analysis...")
r_sample = reddit_df.groupby('Label', group_keys=False).apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 3)))
y_sample = youtube_df.groupby('label', group_keys=False).apply(lambda x: x.sample(min(len(x), SAMPLE_SIZE // 3)))

print(f"✓ Reddit Sample: {len(r_sample)}")
print(f"✓ YouTube Sample: {len(y_sample)}")


## 3. Run Analysis

In [ ]:
# ============================================================================
def process_batch(df, text_col):
    results = []
    texts = df[text_col].tolist()
    
    # Use ThreadPool for faster API calls (be careful with rate limits - 1 QPS usually)
    # We'll use a simple loop with sleep to be safe and respectful of the free tier
    
    print("   Processing comments (this takes time due to API rate limits)...")
    for text in tqdm(texts):
        scores = get_perspective_scores(text)
        if scores:
            results.append(scores)
            time.sleep(1.1) # Sleep to respect ~60 QPM limit
        else:
            results.append({
                'TOXICITY': np.nan, 'SEVERE_TOXICITY': np.nan, 
                'IDENTITY_ATTACK': np.nan, 'INSULT': np.nan, 
                'THREAT': np.nan, 'PROFANITY': np.nan
            })
            
    return pd.DataFrame(results)

print("\nAnalyzing Reddit Data...")
r_scores = process_batch(r_sample, 'self_text')
r_sample = r_sample.reset_index(drop=True)
r_final = pd.concat([r_sample, r_scores], axis=1)

print("\nAnalyzing YouTube Data...")
y_scores = process_batch(y_sample, 'text')
y_sample = y_sample.reset_index(drop=True)
y_final = pd.concat([y_sample, y_scores], axis=1)

# Save raw results


## 4. Visualization & Analysis

In [ ]:
# ============================================================================
print("\n" + "="*80)
print("GENERATING CRITICAL INSIGHTS")
print("="*80)

attributes = ['TOXICITY', 'IDENTITY_ATTACK', 'INSULT', 'THREAT']
labels_map = {'P': 'Pro-Palestine', 'I': 'Pro-Israel', 'N': 'Neutral'}

# 4.1 Platform Comparison (Mean Scores)
print("\n1. Platform Toxicity Comparison")
r_means = r_final[attributes].mean()
y_means = y_final[attributes].mean()

comparison_df = pd.DataFrame({'Reddit': r_means, 'YouTube': y_means})
print(comparison_df)

fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(kind='bar', ax=ax, color=['#FF5722', '#FF0000'], alpha=0.8)
ax.set_title('Average Harmful Content Scores by Platform', fontsize=14, fontweight='bold')
ax.set_ylabel('Perspective API Score (0-1)')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 4.2 Toxicity by Stance (Reddit)
print("\n2. Reddit Toxicity by Stance")
r_final['Label_Full'] = r_final['Label'].map(labels_map)
r_stance_means = r_final.groupby('Label_Full')[attributes].mean()
print(r_stance_means)

fig, ax = plt.subplots(figsize=(12, 6))
r_stance_means.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Reddit: Harmful Content Attributes by Stance', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Score')
plt.legend(title='Attribute')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 4.3 Toxicity by Stance (YouTube)
print("\n3. YouTube Toxicity by Stance")
y_final['Label_Full'] = y_final['Label'].map(labels_map)
y_stance_means = y_final.groupby('Label_Full')[attributes].mean()
print(y_stance_means)

fig, ax = plt.subplots(figsize=(12, 6))
y_stance_means.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('YouTube: Harmful Content Attributes by Stance', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Score')
plt.legend(title='Attribute')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 4.4 Identity Attack Distribution
print("\n4. Identity Attack Analysis")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(x='Label_Full', y='IDENTITY_ATTACK', data=r_final, ax=axes[0], palette='Set2')
axes[0].set_title('Reddit: Identity Attack Scores', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Identity Attack Score')

sns.boxplot(x='Label_Full', y='IDENTITY_ATTACK', data=y_final, ax=axes[1], palette='Set2')
axes[1].set_title('YouTube: Identity Attack Scores', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Identity Attack Score')

plt.tight_layout()
plt.show()


## 5. Generate Report

In [ ]:
# ============================================================================
print("\n" + "="*80)
print("PERSPECTIVE ANALYSIS COMPLETE")
print("="*80)


In [ ]:
from scipy.stats import mannwhitneyu, kruskal

# Mann-Whitney U Tests (Reddit vs YouTube)
print("Mann-Whitney U Tests (Reddit vs YouTube):")
attributes = ['TOXICITY', 'SEVERE_TOXICITY', 'IDENTITY_ATTACK', 'INSULT', 'THREAT', 'PROFANITY']
for attr in attributes:
    if attr in r_final.columns and attr in y_final.columns:
        u_stat, p_val = mannwhitneyu(r_final[attr].dropna(), y_final[attr].dropna())
        print(f"  {attr}: U={u_stat:.0f}, p={p_val:.6f} {'*' if p_val < 0.05 else ''}")

# Kruskal-Wallis Tests (Stance differences)
print("\nKruskal-Wallis Tests (Stance differences):")
for name, df, label_col in [('Reddit', r_final, 'Label'), ('YouTube', y_final, 'Label')]:
    if label_col in df.columns:
        for attr in attributes[:3]:  # Show first 3 attributes
            if attr in df.columns:
                groups = [df[df[label_col] == s][attr].dropna().values for s in ['P', 'I', 'N'] if len(df[df[label_col] == s]) > 0]
                if len(groups) >= 2:
                    h, p = kruskal(*groups)
                    print(f"  {name} – {attr}: H={h:.3f}, p={p:.6f} {'*' if p < 0.05 else ''}")


## Statistical Significance Tests

Mann-Whitney U tests for platform differences and Kruskal-Wallis tests for stance effects.



In [ ]:
# Export toxicity analysis results
print("\n" + "=" * 80)
print("EXPORTING TOXICITY ANALYSIS DATA")
print("=" * 80)

try:
    # Export sampled toxicity data if Perspective API was run
    if 'TOXICITY' in reddit_df.columns:
        reddit_tox_path = OUTPUT_DIR / 'reddit_toxicity_scores.csv'
        youtube_tox_path = OUTPUT_DIR / 'youtube_toxicity_scores.csv'
        
        reddit_df.to_csv(reddit_tox_path, index=False, encoding='utf-8')
        youtube_df.to_csv(youtube_tox_path, index=False, encoding='utf-8')
        
        print(f"\n✅ Exported: {reddit_tox_path}")
        print(f"✅ Exported: {youtube_tox_path}")
    else:
        print("\n⚠️  Toxicity scores not available (Perspective API requires API key)")
        print("📌 Run this notebook locally with API key setup for full toxicity analysis")
        
except Exception as e:
    print(f"\n⚠️  Could not export toxicity data: {e}")

print("\n📌 Toxicity analysis complete. Statistical results displayed above.")


## Export Toxicity Analysis Results

Save toxicity scores and analysis for download.

